# Embedding Route

In [1]:
from utils.llm import get_llm, get_embedding_model
from utils.database.qdrant import QdrantIndexManager

llm = get_llm("llama3.1")
embed_model = get_embedding_model("BAAI/bge-m3")
vector_store = QdrantIndexManager(embed_model=embed_model, index_name="uu").load_vector_index()

c:\Users\arkan\Downloads\University\TA\ta-rag-demo\env\lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


Loaded index 'uu'.


In [2]:
from utils.retriever.uu.embedding_uu import get_embedding_uu

retriever = get_embedding_uu(vector_store=vector_store, llm=llm, embed_model=embed_model, with_self_query=False)

schema_prompt input_variables=['allowed_comparators', 'allowed_operators'] template='<< Structured Request Schema >>\nWhen responding use a markdown code snippet with a JSON object formatted in the following schema:\n\n```json\n{{{{\n    "query": string \\ text string to compare to document contents\n    "filter": string \\ logical condition statement for filtering documents\n}}}}\n```\n\nThe query string should contain only text that is expected to match the contents of documents. Any conditions in the filter should not be mentioned in the query as well.\n\nA logical condition statement is composed of one or more comparison and logical operation statements.\n\nA comparison statement takes the form: `comp(attr, val)`:\n- `comp` ({allowed_comparators}): comparator\n- `attr` (string):  name of attribute to apply the comparison to\n- `val` (string): is the comparison value\n\nA logical operation statement takes the form `op(statement1, statement2, ...)`:\n- `op` ({allowed_operators}): log

In [3]:
retriever.invoke("jawa barat")

[Document(metadata={'title': 'Undang-undang Nomor 10 Tahun 2023 Tentang Provinsi Jawa Barat', 'jenis_bentuk_peraturan': 'UNDANG-UNDANG', 'pemrakarsa': 'PEMERINTAH PUSAT', 'nomor': 10, 'tahun': 2023, 'tentang': 'PROVINSI JAWA BARAT', 'tempat_penetapan': 'Jakarta', 'ditetapkan_tanggal': '04 Mei 2023', 'status': 'Berlaku', 'dokumen_peraturan': 'https://peraturan.go.id/files/uu-no-10-tahun-2023.pdf', 'id': 'hqel0eiu', 'filename': 'uu_04_mei_2023_uu_no_10_tahun_2023_hqel0eiu.pdf', 'page_number': 4, '_id': 'a9a76fe1-9dd8-48aa-8be5-d75d763658aa', '_collection_name': 'uu'}, page_content='NEPUEUT INDONE;IA\n-4-\nt. Kota Sukabumi;\nu. Kota Bandung;\nv. Kota Cirebon;\nw. Kota Bekasi;\nx. Kota Depok;\ny. Kota Cimahi;\nz. Kota Tasikmalaya; dan\naa. Kota Banjar.\nPasal 4\nIbu kota Provinsi Jawa Barat berkedudukan di Kota\nBandung.\nPasal 5\nProvinsi Jawa Barat memiliki karakteristik, yaitu:\na.\ndengan ciri geogralis utama kawasan\ndataran rendah berupa pesisir dan pantai, kawasan\ndataran tinggi be

# Graph Route

In [2]:
from utils.llm import get_llm
from utils.config import get_config

config = get_config()
llm = get_llm("llama3.1")

In [3]:
from utils.retriever.uu.graph_uu import get_graph_uu

retriever = get_graph_uu(llm=llm, config=config)

In [6]:
result = retriever.invoke("Undang Undang No 36 Tahun 1953 dicabut peraturan apa saja?")



> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (p:Peraturan {nomor: '36', tahun: '1953', jenis_peraturan: 'UNDANG-UNDANG'})
MATCH (p)-[:DICABUT]->(revoked:Peraturan)
RETURN 
  p AS revokedRegulation, 
  revoked AS revokingRegulation

Full Context:
[{'revokedRegulation': {'tentang': 'BANK TABUNGAN POS', 'tahun': '1953', 'jenis_peraturan': 'UNDANG-UNDANG', 'tempat_penetapan': 'Jakarta', 'id': 'https://peraturan.go.id/id/uu-no-36-tahun-1953', 'pemrakarsa': 'PEMERINTAH PUSAT', 'nomor': '36', 'status': 'Tidak Berlaku Dicabut Oleh :Undang-Undang Nomor 2 Tahun 1964 Tentang Bank Tabungan Negara'}, 'revokingRegulation': {'tentang': 'BANK TABUNGAN NEGARA', 'tahun': '1964', 'jenis_peraturan': 'UNDANG-UNDANG', 'tempat_penetapan': 'Jakarta', 'id': 'https://peraturan.go.id/id/uu-no-2-tahun-1964', 'pemrakarsa': 'PEMERINTAH PUSAT', 'nomor': '2', 'status': 'Berlaku'}}]

> Finished chain.


In [7]:
result

{'query': 'Undang Undang No 36 Tahun 1953 dicabut peraturan apa saja?',
 'result': 'Pertanyaannya adalah: Undang-Undang No 36 Tahun 1953 dicabut peraturan apa saja?\n\nJawaban dari pertanyaan di atas:\n\nUndang-Undang No 36 Tahun 1953 telah dicabut oleh Undang-Undang Nomor 2 Tahun 1964 Tentang Bank Tabungan Negara. Peraturan yang dicabut adalah tentang Bank Tabungan Pos, yang ditetapkan pada tahun 1953 di Jakarta.\n\nInformasi lebih lanjut tentang peraturan yang dicabut dan mencabutnya adalah sebagai berikut:\n\n* Peraturan yang dicabut: Undang-Undang No 36 Tahun 1953\n* Jenis peraturan: Undang-Undang\n* Tempat penetapan: Jakarta\n* Pemrakarsa: Pemerintah Pusat\n* Nomor: 36\n* Status: Tidak Berlaku Dicabut Oleh : Undang-Undang Nomor 2 Tahun 1964 Tentang Bank Tabungan Negara\n\n* Peraturan yang mencabut: Undang-Undang Nomor 2 Tahun 1964 Tentang Bank Tabungan Negara\n* Jenis peraturan: Undang-Undang\n* Tempat penetapan: Jakarta\n* Pemrakarsa: Pemerintah Pusat\n* Nomor: 2\n* Status: Berla

# All Together

In [1]:
from utils.llm import get_llm, get_embedding_model
from utils.database.qdrant import QdrantIndexManager
from utils.retriever.uu.embedding_uu import get_embedding_uu

llm = get_llm("llama3.1")
embed_model = get_embedding_model("BAAI/bge-m3")
vector_store = QdrantIndexManager(embed_model=embed_model, index_name="uu").load_vector_index()
retriever_uu = get_embedding_uu(vector_store=vector_store, llm=llm, embed_model=embed_model, with_self_query=False)

c:\Users\arkan\Downloads\University\TA\ta-rag-demo\env\lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


Loaded index 'uu'.


In [2]:
from utils.retriever.uu.graph_uu import get_graph_uu
from utils.config import get_config

config = get_config()
graph_chain = get_graph_uu(llm=llm, config=config)

In [3]:
from utils.chain.rag import create_combined_answer_chain

chain_quality = create_combined_answer_chain(
  retriever_uu=retriever_uu,
  graph_chain=graph_chain,
  llm=llm
)

In [5]:
result = chain_quality.invoke({"question": "Undang Undang No 36 Tahun 1953 dicabut peraturan apa saja?", "chat_history": ""})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (p:Peraturan {nomor: '36', tahun: '1953', jenis_peraturan: 'UNDANG-UNDANG'})
MATCH (p)-[:DICABUT]->(revoked:Peraturan)
RETURN 
  p AS revokedRegulation, 
  revoked AS revokingRegulation

Full Context:
[{'revokedRegulation': {'tentang': 'BANK TABUNGAN POS', 'tahun': '1953', 'jenis_peraturan': 'UNDANG-UNDANG', 'tempat_penetapan': 'Jakarta', 'id': 'https://peraturan.go.id/id/uu-no-36-tahun-1953', 'pemrakarsa': 'PEMERINTAH PUSAT', 'nomor': '36', 'status': 'Tidak Berlaku Dicabut Oleh :Undang-Undang Nomor 2 Tahun 1964 Tentang Bank Tabungan Negara'}, 'revokingRegulation': {'tentang': 'BANK TABUNGAN NEGARA', 'tahun': '1964', 'jenis_peraturan': 'UNDANG-UNDANG', 'tempat_penetapan': 'Jakarta', 'id': 'https://peraturan.go.id/id/uu-no-2-tahun-1964', 'pemrakarsa': 'PEMERINTAH PUSAT', 'nomor': '2', 'status': 'Berlaku'}}]

> Finished chain.


In [8]:
print(result['answer'])

Menurut informasi yang tersedia, Undang-undang No. 36 Tahun 1953 dicabut peraturan sebagai berikut:

1. Pasal 180 dihapus.
2. Ketentuan Pasal 185 diubah sehingga berbunyi sebagai berikut: Pasal 185 (1) Pemerintah Pusat dan/atau Pemerintah Daerah dapat memberikan subsidi angkutan pada trayek atau lintas tertentu. (21 Ketentuan lebih lanjut mengenai pemberian subsidi angkutan sebagaimana dimaksud pada ayat (1) diatur dalam Peraturan Pemerintah.
3. Ketentuan Pasal 199 diubah sehingga berbunyi sebagai berikut: Pasal 199 (1) Setiap orang yang melanggar ketentuan sebagaimana dimaksud dalam Pasal 167, Pasal 168, Pasal 173, Pasal 186, Pasal L87, Pasal 189,
4. Ketentuan Pasal 53 diubah sehingga berbunyi sebagai berikut: Pasal 53 (1) Likuidasi Bank dilakukan dengan cara: a. pencairan aset dan/atau penagihan piutang kepada para debitur diikuti dengan pembayaran kewajiban Bank kepada para kreditur dari hasil pencairan dan/atau penagihan; dan/atau b. pengalihan aset dan kewajiban Bank kepada pihak 

In [4]:
result = chain_quality.invoke({"question": "Apa ibukota dari Provinsi Jawa Barat?", "chat_history": ""})

In [5]:
result

{'rewrited question': 'Apa ibukota dari Provinsi Jawa Barat?',
 'context': [Document(metadata={'title': 'Undang-undang Nomor 10 Tahun 2023 Tentang Provinsi Jawa Barat', 'jenis_bentuk_peraturan': 'UNDANG-UNDANG', 'pemrakarsa': 'PEMERINTAH PUSAT', 'nomor': 10, 'tahun': 2023, 'tentang': 'PROVINSI JAWA BARAT', 'tempat_penetapan': 'Jakarta', 'ditetapkan_tanggal': '04 Mei 2023', 'status': 'Berlaku', 'dokumen_peraturan': 'https://peraturan.go.id/files/uu-no-10-tahun-2023.pdf', 'id': 'hqel0eiu', 'filename': 'uu_04_mei_2023_uu_no_10_tahun_2023_hqel0eiu.pdf', 'page_number': 4, '_id': '558b249b-2296-49f3-b2aa-8e96559a714b', '_collection_name': 'uu'}, page_content='Kota Sukabumi;\nt.\n Kota Bandung;\nu.\nKota Cirebon;\nKota Bekasi;\nx.\nKota Depok;\nKota Cimahi;\ny.\n Kota Tasikmalaya; dan\naa. Kota Banjar.\nPasal 4\nIbu kota Provinsi Jawa Barat berkedudukan di Kota\nBandung.\nPasal 5\nProvinsi Jawa Barat memiliki karakteristik, yaitu:.\na. kewilayahan dengan ciri geografis utama kawasan\ndataran 

In [6]:
print(result['answer'])

Berikut adalah jawaban yang komprehensif dan rinci berdasarkan informasi dari satu sumber, yaitu UU (Undang Undang):

Ibu kota Provinsi Jawa Barat berkedudukan di Kota Bandung.

### Referensi:

- [UU](https://peraturan.go.id/files/uu-no-10-tahun-2023.pdf#page=4)

### Daftar Regulasi Unik yang Disebutkan dalam Konteks:
- UU No. 10 Tahun 2023 Tentang Provinsi Jawa Barat
